In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded)
import pandas as pd
df = pd.read_csv('team_stats.csv')

Saving team_stats.csv to team_stats.csv
{'team_stats.csv': b"Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,HomeShots,AwayShots,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards\n2023/24,2024-01-01,Liverpool,Newcastle,4,2,H,0,0,D,34,5,15,3,7,3,16,15,3,5,0,0\n2023/24,2024-01-02,West Ham,Brighton,0,0,D,0,0,D,6,22,2,8,2,0,4,6,1,0,0,0\n2023/24,2024-01-12,Burnley,Luton,1,1,D,1,0,H,13,14,7,5,2,8,7,12,1,0,0,0\n2023/24,2024-01-13,Chelsea,Fulham,1,0,H,1,0,H,17,14,3,4,6,4,13,9,5,1,0,0\n2023/24,2024-01-13,Newcastle,Man City,2,3,A,2,1,H,12,27,5,11,3,13,7,7,1,2,0,0\n2023/24,2024-01-14,Everton,Aston Villa,0,0,D,0,0,D,10,16,2,5,4,5,13,14,1,4,0,0\n2023/24,2024-01-14,Man United,Tottenham,2,2,D,2,1,H,9,16,2,6,8,13,8,5,2,1,0,0\n2023/24,2024-01-20,Arsenal,Crystal Palace,5,0,H,2,0,H,21,12,6,5,6,7,8,12,0,0,0,0\n2023/24,2024-01-20,Br

In [ ]:
df = df.rename(columns={
    "Season": "season",
    "HomeTeam": "home",
    "AwayTeam": "away",
    "FullTimeHomeGoals": "homegoals",
    "FullTimeAwayGoals": "awaygoals",
    "FullTimeResult": "result",
    "HomeShots": "homeshots",
    "AwayShots": "awayshots",
    "HomeShotsOnTarget": "homeshotstarget",
    "AwayShotsOnTarget": "awayshotstarget",
    "HomeYellowCards": "homeyellow",
    "AwayYellowCards": "awayyellow",
    "HomeRedCards": "homered",
    "AwayRedCards": "awayred"
})


In [ ]:
home = df[['season','home','homegoals','awaygoals','result']].copy()
home.columns = ['season','team','goalwin','goallose','result']
home['points']=home['result'].map({'H': 3, 'A':0 , 'D':1})

away = df[['season','away','awaygoals','homegoals','result']].copy()
away.columns = ['season','team','goalswin','goallose','result']
away['points']= home['result'].map({'H':0 , 'D': 1 , 'A': 3 })

gopbang = pd.concat([home,away],ignore_index = True)

gopbang.groupby(['season','team']).agg({'points':'sum'}).reset_index().sort_values('points',ascending = False)


,season,team,points
31,2024/25,Liverpool,82
20,2024/25,Arsenal,67
32,2024/25,Man City,64
34,2024/25,Newcastle,63
25,2024/25,Chelsea,63
35,2024/25,Nott'm Forest,61
21,2024/25,Aston Villa,60
22,2024/25,Bournemouth,53
24,2024/25,Brighton,52
23,2024/25,Brentford,52


In [ ]:
hieusuat = ( df.groupby(['season'], as_index = False).agg({ 'homeshots':'sum',
                                                           'awayshots':'sum',
                                                            'homeshotstarget':'sum',
                                                            'awayshotstarget':'sum'}))
hieusuat['hieusuatnha']= hieusuat['homeshotstarget'] /hieusuat['homeshots']
hieusuat['hieusuatkhach']= hieusuat['awayshotstarget']/hieusuat['awayshots']
hieusuat[['season','hieusuatnha','hieusuatkhach']]


,season,hieusuatnha,hieusuatkhach
0,2023/24,0.356396,0.375326
1,2024/25,0.351452,0.349788


In [ ]:
tong_the = (
  pd.concat([
    df[['home','homeyellow','homered']].rename( columns={'home' : 'team', 'homeyellow': 'yellow', 'homered': 'red'}),
    df[['away','awayyellow','awayred']].rename( columns ={'away' : 'team', 'awayyellow': 'yellow', 'awayred': 'red'})
  ]).groupby(['team'], as_index = True)[['yellow','red']].mean()
     .rename(columns={'yellow':'yellow_mean','red':'red_mean'})

)
tong_the.sort_values('yellow_mean', ascending=False).head(100).reset_index()

,team,yellow_mean,red_mean
0,Chelsea,2.584906,0.037736
1,Bournemouth,2.481481,0.074074
2,Southampton,2.457143,0.085714
3,Ipswich,2.314286,0.142857
4,Leicester,2.314286,0.000000
5,Wolves,2.207547,0.056604
6,Everton,2.094340,0.037736
7,Nott'm Forest,2.037736,0.037736
8,Man United,2.037736,0.056604
9,Fulham,2.037736,0.056604


In [ ]:
homee = df.copy()
homee['win']= (homee['result']=='H').astype(int)
homewinrate = homee.groupby('home')['win'].mean().reset_index()
homewinrate.rename(columns={'home': 'team', 'win': 'winhome'}, inplace=True)

awayy = df.copy()
awayy['win']=(awayy['result']=='A').astype(int)
awaywinrate = awayy.groupby('away')['win'].mean().reset_index()
awaywinrate.rename(columns={'away': 'team', 'win': 'winaway'}, inplace=True)

winrate = pd.merge(homewinrate, awaywinrate, on='team', how='outer').fillna(0)

winrate.head(10)






,team,winhome,winaway
0,Arsenal,0.666667,0.615385
1,Aston Villa,0.481481,0.423077
2,Bournemouth,0.407407,0.333333
3,Brentford,0.407407,0.333333
4,Brighton,0.370370,0.259259
5,Burnley,0.111111,0.111111
6,Chelsea,0.666667,0.384615
7,Crystal Palace,0.407407,0.307692
8,Everton,0.333333,0.153846
9,Fulham,0.384615,0.370370


In [16]:
df['homedraw'] = df['result'].map({'H': 0, 'D': 1, 'A': 0})
df['awaydraw'] = df['result'].map({'H': 0, 'D': 1, 'A': 0})

HomeDraw = df.groupby(['season','home'])['homedraw'].sum()
AwayDraw = df.groupby(['season','away'])['awaydraw'].sum()

TotalDraw = HomeDraw.add(AwayDraw, fill_value=0).reset_index()
TotalDraw.columns = ['season','team','totaldraw']

MaxDraw = TotalDraw.groupby('season')['totaldraw'].max().reset_index()
MaxDraw.columns = ['season','maxdraw']

Result = TotalDraw.merge(MaxDraw, on='season')
print(Result[Result['totaldraw'] == Result['maxdraw']].sort_values(['season','totaldraw'], ascending=[False,False]))


     season      team  totaldraw  maxdraw
27  2024/25   Everton         15       15
5   2023/24   Burnley          7        7
8   2023/24   Everton          7        7
18  2023/24  West Ham          7        7
